# ETL — VRA/ANAC 2019

Este notebook executa o pipeline ETL referente somente a 2019. A implementação das regras está centralizada em `etl_vra_anac_2019.py`; o notebook prepara o ambiente, configura os caminhos, executa o script e confere os resultados no PostgreSQL.

## Estrutura esperada

```text
Projeto-Integracao-ANAC/
├── etl_vra_anac_2019.py
├── etl_vra_anac_2019.ipynb
├── requirements.txt
├── .env
└── dados/
    ├── brutos/2019/       # VRA_2019_01.csv ... VRA_2019_12.csv
    └── processados/2019/  # resultados do ETL
```

**PostgreSQL local:** execute este notebook no VS Code/Jupyter no mesmo computador do PostgreSQL. **Google Colab:** a máquina virtual do Colab não acessa o `localhost` do seu computador; para executar a carga no Colab, configure um PostgreSQL remoto acessível e adicione as credenciais em Secrets. Não coloque senhas no notebook ou no GitHub.

A regra atual remove duplicatas exatas após a padronização, comparando os 20 campos canônicos em todos os 12 meses. A primeira ocorrência é mantida; as excedentes são exportadas para `duplicatas_removidas.csv.gz`.


In [ ]:
# Instala as dependências essenciais do pipeline (também estão em requirements.txt).
%pip install -q "pandas>=2.2,<3.0" "SQLAlchemy>=2.0,<3.0" "psycopg[binary]>=3.2,<4.0" "python-dotenv>=1.0,<2.0"


In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from dotenv import load_dotenv

# Detecta o Google Colab. No VS Code/Jupyter, usa a pasta atual do projeto.
try:
    from google.colab import drive, userdata
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Projeto-Integracao-ANAC')
except ImportError:
    IN_COLAB = False
    PROJECT_DIR = Path.cwd()

# Ajuste PROJECT_DIR se o notebook estiver em outra pasta.
print('Ambiente:', 'Google Colab' if IN_COLAB else 'VS Code/Jupyter local')
print('Pasta do projeto:', PROJECT_DIR)
assert PROJECT_DIR.exists(), f'Pasta do projeto não encontrada: {PROJECT_DIR}'

# Carrega variáveis do .env local; no Colab, usa os Secrets configurados pelo usuário.
load_dotenv(PROJECT_DIR / '.env', override=False)
if IN_COLAB:
    for key in ['PGHOST', 'PGPORT', 'PGDATABASE', 'PGUSER', 'PGPASSWORD', 'PGSCHEMA', 'DATABASE_URL']:
        try:
            secret_value = userdata.get(key)
            if secret_value:
                os.environ[key] = secret_value
        except Exception:
            pass

INPUT_DIR = PROJECT_DIR / 'dados' / 'brutos' / '2019'
OUTPUT_DIR = PROJECT_DIR / 'dados' / 'processados' / '2019'
SCRIPT_PATH = PROJECT_DIR / 'etl_vra_anac_2019.py'

print('Entrada:', INPUT_DIR)
print('Saída:', OUTPUT_DIR)
assert SCRIPT_PATH.exists(), f'Script não encontrado: {SCRIPT_PATH}'


## 1. Conferência dos arquivos de origem

O pipeline exige os 12 CSVs mensais de 2019 com nomes `VRA_2019_MM.csv` e o cabeçalho de 20 colunas esperado no contrato.


In [ ]:
arquivos = sorted(INPUT_DIR.glob('VRA_2019_*.csv'))
print(f'Arquivos encontrados: {len(arquivos)}')
for arquivo in arquivos:
    print(arquivo.name, f'({arquivo.stat().st_size:,} bytes)')
assert len(arquivos) == 12, 'A pasta dados/brutos/2019 deve conter os 12 CSVs mensais.'


## 2. Configuração do PostgreSQL

No ambiente local, o script lê as variáveis do arquivo `.env` na raiz do projeto. No Colab, configure os mesmos nomes em **Secrets** (com acesso habilitado para o notebook). O banco `vra_dw_2019` deve existir antes da execução.

O schema padrão é `dw_etl`. A senha não é exibida nesta célula.


In [ ]:
required = ['PGHOST', 'PGPORT', 'PGDATABASE', 'PGUSER', 'PGPASSWORD']
if not os.getenv('DATABASE_URL'):
    missing = [key for key in required if not os.getenv(key)]
    assert not missing, f'Configure no .env/Secrets estas variáveis: {", ".join(missing)}'

print('Host:', os.getenv('PGHOST', '(definido em DATABASE_URL)'))
print('Porta:', os.getenv('PGPORT', '(definida em DATABASE_URL)'))
print('Banco:', os.getenv('PGDATABASE', '(definido em DATABASE_URL)'))
print('Usuário:', os.getenv('PGUSER', '(definido em DATABASE_URL)'))
print('Schema:', os.getenv('PGSCHEMA', 'dw_etl'))
print('Senha configurada:', 'sim' if (os.getenv('PGPASSWORD') or os.getenv('DATABASE_URL')) else 'não')


## 3. Execução do ETL

A opção `REPLACE_CARGA_EXISTENTE` é `False` por segurança. Caso as tabelas do ETL já existam no schema, o script interrompe a execução. Só altere para `True` quando desejar apagar e recriar as tabelas ETL desse schema.


In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
from etl_vra_anac_2019 import run_etl

REPLACE_CARGA_EXISTENTE = False  # True apaga/recria as tabelas ETL do schema dw_etl
resultado = run_etl(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    load_database=True,
    replace=REPLACE_CARGA_EXISTENTE,
)
print(json.dumps(resultado, ensure_ascii=False, indent=2, default=str))


## 4. Conferência das tabelas no PostgreSQL

Esta célula consulta as contagens carregadas e verifica as relações entre a fato e as dimensões. O script também valida as chaves estrangeiras durante a transação de carga.


In [ ]:
from sqlalchemy import text
from etl_vra_anac_2019 import _postgres_engine

schema = os.getenv('PGSCHEMA', 'dw_etl')
tabelas = [
    'dim_tempo', 'dim_companhia_aerea', 'dim_aeroporto', 'dim_situacao_voo',
    'dim_justificativa', 'dim_tipo_linha', 'dim_equipamento', 'fato_voo',
    'rejeicoes', 'log_carga_arquivo'
]
engine = _postgres_engine()
try:
    with engine.connect() as con:
        contagens = pd.DataFrame([
            {'tabela': tabela, 'registros': con.execute(
                text(f'SELECT COUNT(*) FROM "{schema}"."{tabela}"')
            ).scalar_one()}
            for tabela in tabelas
        ])
    display(contagens)
finally:
    engine.dispose()


## 5. Arquivos gerados e evidência da execução

A pasta `dados/processados/2019/` recebe staging mensal, exportações das dimensões e da fato, rejeições, duplicatas removidas, log por arquivo, qualidade por coluna e `etl_resumo.json`.


In [ ]:
resumo_path = OUTPUT_DIR / 'etl_resumo.json'
assert resumo_path.exists(), 'O arquivo etl_resumo.json não foi gerado.'
resumo = json.loads(resumo_path.read_text(encoding='utf-8'))
print(json.dumps(resumo, ensure_ascii=False, indent=2))

arquivos_gerados = sorted(p.name for p in OUTPUT_DIR.iterdir())
print('\nItens na pasta de saída:')
for nome in arquivos_gerados:
    print('-', nome)


## 6. Limite do escopo

Este notebook implementa somente o ETL de 2019. Ele não implementa ELT nem integra 2020 e 2021. Essas partes devem ser realizadas e reconciliadas pelo grupo em etapas separadas.
